In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("clean_data.csv")

In [3]:
df['date'] = pd.to_datetime(df['date'],format='%Y-%m-%d')
df = df.set_index('date')

In [4]:
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 300259 entries, 2022-02-11 to 2022-03-31
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   airline              300259 non-null  str  
 1   from                 300259 non-null  str  
 2   stop                 300259 non-null  str  
 3   to                   300259 non-null  str  
 4   price                300259 non-null  int64
 5   class                300259 non-null  str  
 6   time_of_day          300259 non-null  str  
 7   is_weekday           300259 non-null  int64
 8   month                300259 non-null  int64
 9   total_duration_mins  300259 non-null  int64
dtypes: int64(4), str(6)
memory usage: 37.7 MB


In [5]:
df.sample(6)

,airline,from,stop,to,price,class,time_of_day,is_weekday,month,total_duration_mins
date,,,,,,,,,,
2022-03-09,Vistara,Kolkata,1-stop,Mumbai,59254,business,early morning,2,3,865
2022-03-09,Vistara,Delhi,1-stop,Bangalore,3672,economy,afternoon,2,3,285
2022-03-16,SpiceJet,Kolkata,1-stop,Mumbai,5934,economy,night,2,3,645
2022-03-20,Vistara,Kolkata,1-stop,Bangalore,52287,business,morning,6,3,630
2022-02-24,Air India,Chennai,1-stop,Delhi,45185,business,early morning,3,2,780
2022-03-21,AirAsia,Delhi,1-stop,Hyderabad,2050,economy,early morning,0,3,585


In [6]:
x = df.drop(['price','total_duration_mins'],axis=1)
y = df[['total_duration_mins','price']]

In [7]:
from sklearn.multioutput import RegressorChain
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from xgboost import XGBRegressor # 'objective':'count:poisson' to avoid negative values
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error,root_mean_squared_error

In [8]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [9]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(240207, 8) (60052, 8) (240207, 2) (60052, 2)


In [10]:
test = pd.concat([x_test,y_test],axis=1)
test.to_csv("test.csv")

In [11]:
from sklearn.compose import ColumnTransformer

col = df.select_dtypes(exclude='int').columns
ohe = OneHotEncoder(drop='first')

ct = ColumnTransformer([('ohe',ohe,col)],remainder='passthrough')
x_train = ct.fit_transform(x_train)
x_test = ct.transform(x_test)

In [12]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(240207, 27) (60052, 27) (240207, 2) (60052, 2)


In [13]:
import optuna

In [14]:

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0), 
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),    
        'gamma': trial.suggest_float('gamma', 1e-3, 10.0, log=True),
        'random_state': 42,
        'objective':'count:poisson',
        'n_jobs': -1  #
    }

    model = XGBRegressor(**params)
    model = RegressorChain(model,order=[0,1])
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    score = r2_score(y_test, y_pred)
    return score


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=15)

print('Best hyperparameters:', study.best_params)
print('Best score:', study.best_value)

[I 2026-06-30 13:28:41,329] A new study created in memory with name: no-name-fbcdaae2-cf50-4db7-b22c-e87744dc9225
[I 2026-06-30 13:29:10,379] Trial 0 finished with value: 0.7576961253142005 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.05395030966670229, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'gamma': 0.0017073967431528124}. Best is trial 0 with value: 0.7576961253142005.
[I 2026-06-30 13:30:02,505] Trial 1 finished with value: 0.7579205575917765 and parameters: {'n_estimators': 450, 'max_depth': 7, 'learning_rate': 0.051059032093947576, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971, 'min_child_weight': 9, 'gamma': 0.0070689749506246055}. Best is trial 1 with value: 0.7579205575917765.
[I 2026-06-30 13:30:10,747] Trial 2 finished with value: 0.6289261538355431 and parameters: {'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.02014847788415866, 'subsample': 0.76237821581

Best hyperparameters: {'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.09076566077842713, 'subsample': 0.628455983614455, 'colsample_bytree': 0.7837181193173234, 'min_child_weight': 8, 'gamma': 0.008101324842116018}
Best score: 0.7636089714678895


In [15]:
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("flight_price")

model = XGBRegressor(**study.best_params)
model = RegressorChain(model,order=[0,1])
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
scores = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

with mlflow.start_run(run_name="XGBRegressor_RegressorChain_model"):
    mlflow.log_params(study.best_params)
    mlflow.log_metric("r2_score", scores)
    mlflow.log_metric("mean_absolute_error", mae)
    mlflow.log_metric("mean_squared_error", mse)
    mlflow.log_metric("root_mean_squared_error", rmse)
    
    mlflow.sklearn.log_model(model, artifact_path="XGBRegressor_RegressorChain_model") 

2026/06/30 13:33:38 INFO mlflow.tracking.fluent: Experiment with name 'flight_price' does not exist. Creating a new experiment.
2026/06/30 13:34:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 13:34:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBRegressor_RegressorChain_model at: http://127.0.0.1:5000/#/experiments/1/runs/3fc1d935c2ca491382f196d115cd373c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [16]:
import joblib
joblib.dump(model,'XGBRegressor_RegressorChain_model.joblib')
joblib.dump(ct,'ct.joblib')

['ct.joblib']

In [17]:

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=50), 
        'max_depth': trial.suggest_int('max_depth', 3, 10),                
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.5, 0.9),       
        'max_samples': trial.suggest_float('max_samples', 0.5, 0.8),        
        'n_jobs': -1,                                                        
        'random_state': 42
    }

    model = RandomForestRegressor(**params)
    model = RegressorChain(model, order=[0, 1])
    
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    score = r2_score(y_test, y_pred)
    
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))

study.optimize(objective, n_trials=15)

print('Best hyperparameters:', study.best_params)
print('Best score:', study.best_value)


[I 2026-06-30 13:34:20,162] A new study created in memory with name: no-name-85e6cf3b-d850-464a-b43d-64f639bb27ea
[I 2026-06-30 13:36:02,076] Trial 0 finished with value: 0.7376802252804022 and parameters: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 0.5624074561769746, 'max_samples': 0.5467983561008608}. Best is trial 0 with value: 0.7376802252804022.
[I 2026-06-30 13:36:22,852] Trial 1 finished with value: 0.7307392552622488 and parameters: {'n_estimators': 50, 'max_depth': 9, 'min_samples_split': 13, 'min_samples_leaf': 15, 'max_features': 0.508233797718321, 'max_samples': 0.7909729556485983}. Best is trial 0 with value: 0.7376802252804022.
[I 2026-06-30 13:37:11,889] Trial 2 finished with value: 0.6836340264567622 and parameters: {'n_estimators': 450, 'max_depth': 4, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 0.621696897183815, 'max_samples': 0.6574269294896714}. Best is trial 0 with value: 0.73768022528

Best hyperparameters: {'n_estimators': 50, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 14, 'max_features': 0.6246844304357644, 'max_samples': 0.6560204063533432}
Best score: 0.7378452769644983


In [18]:
model = RandomForestRegressor(**study.best_params)
model = RegressorChain(model,order=[0,1])
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
scores = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

with mlflow.start_run(run_name="RandomForestRegressor_RegressorChain_model"):
    mlflow.log_params(study.best_params)
    mlflow.log_metric("r2_score", scores)
    mlflow.log_metric("mean_absolute_error", mae)
    mlflow.log_metric("mean_squared_error", mse)
    mlflow.log_metric("root_mean_squared_error", rmse)
    mlflow.sklearn.log_model(model, artifact_path="RandomForestRegressor_RegressorChain_model")


2026/06/30 13:52:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/30 13:52:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForestRegressor_RegressorChain_model at: http://127.0.0.1:5000/#/experiments/1/runs/4ae6a6d2b2ed49bfb24113b9e13708f5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:
joblib.dump(model,'RandomForestRegressor_RegressorChain_model.joblib')

['RandomForestRegressor_RegressorChain_model.joblib']